In [43]:
from pathlib import Path
import sys
import time

# Adding to path
#PROJECT_ROOT = Path("/home/lsito/mast") 
PROJECT_ROOT = Path("c:/Users/Leonardo/Desktop/Mast/mast") # On my Windows Laptop

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [45]:
from serial.tools import list_ports

for port in list_ports.comports():
    print("device:", port.device)
    print("description:", port.description)
    print("manufacturer:", port.manufacturer)
    print("hwid:", port.hwid)
    print()

device: COM3
description: Standard Serial over Bluetooth link (COM3)
manufacturer: Microsoft
hwid: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0046\9&20FE5BE0&0&6C06D631205D_C00000000

device: COM4
description: Standard Serial over Bluetooth link (COM4)
manufacturer: Microsoft
hwid: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0000\9&20FE5BE0&0&000000000000_00000000



In [42]:
import time
import serial

PORT = "COM5"

BAUD_RATES = [
    9600,
    19200,
    38400,
    57600,
    115200,
]

COMMANDS = [
    "*IDN?",
    "IDN?",
    "POS? 1",
    "SVO? 1",
    "ERR?",
]

LINE_ENDINGS = [
    "\n",
    "\r\n",
    "\r",
]


def try_command(port, baudrate, command, ending):
    """
    Try one serial command and return the decoded response.
    """
    try:
        with serial.Serial(
            port=port,
            baudrate=baudrate,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=1.0,
            write_timeout=1.0,
            rtscts=False,
            dsrdtr=False,
            xonxoff=False,
        ) as ser:
            ser.reset_input_buffer()
            ser.reset_output_buffer()

            ser.write((command + ending).encode("ascii"))
            ser.flush()

            time.sleep(0.3)

            data = ser.read_all()

        return data.decode("ascii", errors="replace").strip()

    except Exception as exc:
        return f"ERROR: {exc}"


found = None

for baudrate in BAUD_RATES:
    print()
    print("Trying baudrate:", baudrate)

    for command in COMMANDS:
        for ending in LINE_ENDINGS:
            response = try_command(PORT, baudrate, command, ending)

            if response and not response.startswith("ERROR"):
                found = {
                    "port": PORT,
                    "baudrate": baudrate,
                    "command": command,
                    "ending": ending,
                    "response": response,
                }

                print("FOUND RESPONSE")
                print(found)
                break

        if found is not None:
            break

    if found is not None:
        break

if found is None:
    print("No response found.")


Trying baudrate: 9600

Trying baudrate: 19200

Trying baudrate: 38400

Trying baudrate: 57600

Trying baudrate: 115200
No response found.


In [32]:
try:
    ser.close()
    print("Closed previous serial object")
except Exception as exc:
    print("No previous serial object to close:", exc)

ser = serial.Serial(
    port=COM_PORT,
    baudrate=BAUDRATE,
    bytesize=serial.EIGHTBITS,
    parity=serial.PARITY_NONE,
    stopbits=serial.STOPBITS_ONE,
    timeout=1.0,
    write_timeout=1.0,
)

Closed previous serial object


In [33]:
def send(command):
    message = command.strip() + "\n"
    ser.write(message.encode("ascii"))
    ser.flush()


def query(command, delay=0.05):
    send(command)
    time.sleep(delay)
    response = ser.read_until(b"\n").decode("ascii", errors="replace").strip()
    return response


def query_all(command, delay=0.05):
    send(command)
    time.sleep(delay)
    data = ser.read_all().decode("ascii", errors="replace").strip()
    return data

In [39]:
import time
import serial

PORT = "COM5"
BAUDRATE = 115200

try:
    ser.close()
    print("Closed previous serial object")
except Exception as exc:
    print("No previous serial object to close:", exc)


baud_rates = [9600, 19200, 38400, 57600, 115200]
commands = ["*IDN?", "IDN?", "ERR?", "POS? 1", "SVO? 1"]
endings = ["\n", "\r\n", "\r"]

for baudrate in baud_rates:
    print("\nBAUDRATE", baudrate)

    try:
        with serial.Serial(
            port=PORT,
            baudrate=baudrate,
            bytesize=serial.EIGHTBITS,
            parity=serial.PARITY_NONE,
            stopbits=serial.STOPBITS_ONE,
            timeout=1.0,
            write_timeout=1.0,
            rtscts=False,
            dsrdtr=False,
            xonxoff=False,
        ) as ser:
            for command in commands:
                for ending in endings:
                    ser.reset_input_buffer()
                    ser.reset_output_buffer()

                    ser.write((command + ending).encode("ascii"))
                    ser.flush()

                    time.sleep(0.3)

                    data = ser.read_all()

                    if data:
                        print("FOUND")
                        print("baudrate:", baudrate)
                        print("command:", repr(command))
                        print("ending:", repr(ending))
                        print("raw:", data)
                        print("decoded:", data.decode("ascii", errors="replace"))
                        raise SystemExit

    except Exception as exc:
        print("failed:", exc)

print("No response found.")

Closed previous serial object

BAUDRATE 9600

BAUDRATE 19200

BAUDRATE 38400

BAUDRATE 57600

BAUDRATE 115200
No response found.


In [40]:
from serial.tools import list_ports

for port in list_ports.comports():
    print(port.device)
    print("  description:", port.description)
    print("  manufacturer:", port.manufacturer)
    print("  hwid:", port.hwid)
    print()

COM3
  description: Standard Serial over Bluetooth link (COM3)
  manufacturer: Microsoft
  hwid: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0046\9&20FE5BE0&0&6C06D631205D_C00000000

COM4
  description: Standard Serial over Bluetooth link (COM4)
  manufacturer: Microsoft
  hwid: BTHENUM\{00001101-0000-1000-8000-00805F9B34FB}_LOCALMFG&0000\9&20FE5BE0&0&000000000000_00000000

COM5
  description: USB Serial Port (COM5)
  manufacturer: FTDI
  hwid: USB VID:PID=0403:6001 SER=FTU8HBRQA



In [26]:
def print_motor_status(label=""):
    status = motor.status()

    if label:
        print(label)

    print("Connected:", status.connected)
    print("Position:", status.current_position)
    print("Target:", status.target_position)
    print("Target reached:", status.target_reached)
    print("Is moving:", status.is_moving)
    print("Servo on:", status.servo_on)
    print()

    return status


print_motor_status("Current motor status")

Current motor status
Connected: True
Position: 0.0
Target: 0.0
Target reached: True
Is moving: False
Servo on: True



MotorStatus(connected=True, current_position=0.0, target_position=0.0, target_reached=True, is_moving=False, servo_on=True, minimum_position=None, maximum_position=None, controller_id='BYPASS PI motor controller')

In [27]:
if RUN_MOTION:
    print(f"Moving relative by {MOVE_RELATIVE}")
    motor.move_relative(MOVE_RELATIVE)

    for _ in range(20):
        status = print_motor_status("Moving...")
        if status.target_reached and not status.is_moving:
            break
        time.sleep(0.3)

    print_motor_status("Final status")
else:
    print("RUN_MOTION is False, so no movement was executed.")

Moving relative by 10.0
Moving...
Connected: True
Position: 10.0
Target: 10.0
Target reached: True
Is moving: False
Servo on: True

Final status
Connected: True
Position: 10.0
Target: 10.0
Target reached: True
Is moving: False
Servo on: True



In [28]:
if RUN_MOTION:
    print(f"Moving absolute to {MOVE_ABSOLUTE_2}")
    motor.move_absolute(MOVE_ABSOLUTE_2)

    for _ in range(20):
        status = print_motor_status("Moving...")
        if status.target_reached and not status.is_moving:
            break
        time.sleep(0.3)

    print_motor_status("Final status")
else:
    print("RUN_MOTION is False, so no absolute movement was executed.")

Moving absolute to 0.0
Moving...
Connected: True
Position: 0.0
Target: 0.0
Target reached: True
Is moving: False
Servo on: True

Final status
Connected: True
Position: 0.0
Target: 0.0
Target reached: True
Is moving: False
Servo on: True



In [29]:
try:
    status = motor.stop()
    print("STOP sent.")
    print("Servo on:", status.servo_on)
    print("Position:", status.current_position)
except Exception as exc:
    print("STOP failed:", exc)

STOP sent.
Servo on: False
Position: 0.0


In [12]:
try:
    motor.close()
    print("Motor communication closed.")
except Exception as exc:
    print("Close failed:", exc)

Motor communication closed.
